<a href="https://colab.research.google.com/github/yassiiirrrr/Short-Term-Solar-Energy-Generation-Forecasting-Using-Weather-Data-and-Machine-Learning-Model/blob/main/notebooks/02_Ensemble_Baselines_XGBoost_RF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# 1. Import Required Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# 2. Load the Dataset
# IMPORTANT: Upload your Kaggle dataset to Colab and replace 'solar_data.csv' with your actual file name
file_path = 'train.csv'
try:
    df = pd.read_csv(file_path)
    print("Data loaded successfully!")
except FileNotFoundError:
    print(f"Error: Could not find {file_path}. Please upload your Kaggle dataset.")

# 3. Data Preprocessing & Feature Engineering
# Assuming your dataset has a datetime column named 'DATE_TIME' and the target is 'DC_POWER' or 'AC_POWER'
# Update 'target_column' and 'datetime_column' to match your specific Kaggle dataset headers
datetime_column = 'timestamp'
target_column = 'Active_Power'

if datetime_column in df.columns:
    # Convert to datetime object and extract time-based features
    df[datetime_column] = pd.to_datetime(df[datetime_column])
    df['hour'] = df[datetime_column].dt.hour
    df['month'] = df[datetime_column].dt.month
    df['day_of_year'] = df[datetime_column].dt.dayofyear

    # Drop the original datetime column as models require numerical inputs
    df = df.drop(columns=[datetime_column])

# Drop any missing values to ensure clean training data
df = df.dropna()

# Separate features (X) and target variable (y)
# Also dropping irrelevant categorical columns (like PLANT_ID or SOURCE_KEY) if they exist
columns_to_drop = [target_column, 'PLANT_ID', 'SOURCE_KEY']
X = df.drop(columns=[col for col in columns_to_drop if col in df.columns])
y = df[target_column]

# Split the data into 80% training and 20% testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

# 4. Initialize Baseline Models
# Random Forest
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

# XGBoost
xgb_model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, learning_rate=0.1, random_state=42)

# 5. Train Models
print("\nTraining Random Forest...")
rf_model.fit(X_train, y_train)

print("Training XGBoost...")
xgb_model.fit(X_train, y_train)

# 6. Generate Predictions
rf_predictions = rf_model.predict(X_test)
xgb_predictions = xgb_model.predict(X_test)

# 7. Evaluate Models (Calculating R², RMSE, and MAE)
def evaluate_model(model_name, y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    print(f"--- {model_name} Performance ---")
    print(f"R² Score: {r2:.4f}")
    print(f"RMSE:     {rmse:.4f}")
    print(f"MAE:      {mae:.4f}\n")

print("\n=== Experimental Results ===")
evaluate_model("Random Forest", y_test, rf_predictions)
evaluate_model("XGBoost", y_test, xgb_predictions)

Data loaded successfully!
Training data shape: (153900, 12)
Testing data shape: (38475, 12)

Training Random Forest...
Training XGBoost...

=== Experimental Results ===
--- Random Forest Performance ---
R² Score: 0.9906
RMSE:     0.7419
MAE:      0.3009

--- XGBoost Performance ---
R² Score: 0.9855
RMSE:     0.9208
MAE:      0.4225

